## U-Net

In [82]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms.functional


In [81]:
test.weight

Parameter containing:
tensor([[[[-0.1865,  0.1912, -0.1332],
          [ 0.3171,  0.1661,  0.0524],
          [ 0.2981, -0.3218, -0.0857]]]], requires_grad=True)

In [ ]:

class DoubleConv(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=3, padding=1)
        self.act1 = nn.ReLU()

        self.conv2 = nn.Conv2d(in_channels=out_channels, out_channels=out_channels, kernel_size=3, padding=1)
        self.act2 = nn.ReLU()
    
    def forward(self, x):

        x = self.conv1(x)
        x = self.act1(x)
        x = self.conv2(x)
        x = self.act2(x)
        return x


class CropAndConcat(nn.Module):

    def forward(self, x: torch.Tensor, contracting_x: torch.Tensor):

        contracting_x = torchvision.transforms.functional.center_crop(contracting_x, (x.shape[2], x.shape[3]))
        x = torch.cat([x, contracting_x], dim=1)

        return x

class Upsample(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.up_conv = nn.ConvTranspose2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=2,
            stride=2
        )

    def forward(self, x):
        return self.up_conv(x)


class UNet(nn.Module):

    def __init__(self, in_channels, out_channels):
        
        self.down_conv = nn.ModuleList([DoubleConv(i, o) for (i, o) in
                                        [(in_channels, 64), (64, 128), (128, 256), (256, 512)]])
        self.down_sample = nn.ModuleList([nn.MaxPool2d(2) for _ in range(4)])

        self.middle_conv = DoubleConv(512, 1024)

        self.up_sample = nn.ModuleList([Upsample(i, o) for (i, o) in 
                                        [(1024, 512), (512, 256), (256, 128), (128, 64)]])
        self.up_conv = nn.ModuleList([DoubleConv(i, o) for (i, o) in
                                        [(1024, 512), (512, 256), (256, 128), (128, 64)]])

        self.crop_and_concat = CropAndConcat()

        self.final_conv = DoubleConv(64, out_channels=out_channels)

    def forward(self, x):

        pass_through = []
        for i,conv in enumerate(self.down_conv):

            x = conv(x)
            pass_through.append(x)
            x = self.down_sample[i](x)
        
        x = self.middle_conv(x)

        for i,up in enumerate(self.up_sample):

            x = up(x)
            x = self.crop_and_concat(x, pass_through.pop())
            x = self.up_conv[i](x)
        
        x = self.final_conv(x)
        return x




        

        



_IncompleteInputError: incomplete input (3983762426.py, line 1)